In [1]:
import mlflow
import pandas as pd
import mlflow.sklearn
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score
import pandas as pd
import re
import string
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer
import numpy as np

/opt/anaconda3/envs/atlas/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
df = pd.read_csv('IMDB.csv')
df = df.sample(500)
df.to_csv('data.csv', index=False)
df.head()

,review,sentiment
612,Viva Variety was a unique hybrid program that ...,positive
86,Two years ago at Sundance I loved Josh Kornblu...,negative
662,I would reccomend this film to everyone. Not o...,positive
965,This movie is a true reflection of the Austral...,positive
871,Legend has it that at the gala Hollywood premi...,negative


In [3]:
# data preprocessing

# Define text preprocessing functions
def lemmatization(text):
    """Lemmatize the text."""
    lemmatizer = WordNetLemmatizer()
    text = text.split()
    text = [lemmatizer.lemmatize(word) for word in text]
    return " ".join(text)

def remove_stop_words(text):
    """Remove stop words from the text."""
    stop_words = set(stopwords.words("english"))
    text = [word for word in str(text).split() if word not in stop_words]
    return " ".join(text)

def removing_numbers(text):
    """Remove numbers from the text."""
    text = ''.join([char for char in text if not char.isdigit()])
    return text

def lower_case(text):
    """Convert text to lower case."""
    text = text.split()
    text = [word.lower() for word in text]
    return " ".join(text)

def removing_punctuations(text):
    """Remove punctuations from the text."""
    text = re.sub('[%s]' % re.escape(string.punctuation), ' ', text)
    text = text.replace('؛', "")
    text = re.sub('\s+', ' ', text).strip()
    return text

def removing_urls(text):
    """Remove URLs from the text."""
    url_pattern = re.compile(r'https?://\S+|www\.\S+')
    return url_pattern.sub(r'', text)

def normalize_text(df):
    """Normalize the text data."""
    try:
        df['review'] = df['review'].apply(lower_case)
        df['review'] = df['review'].apply(remove_stop_words)
        df['review'] = df['review'].apply(removing_numbers)
        df['review'] = df['review'].apply(removing_punctuations)
        df['review'] = df['review'].apply(removing_urls)
        df['review'] = df['review'].apply(lemmatization)
        return df
    except Exception as e:
        print(f'Error during text normalization: {e}')
        raise

In [6]:
df = normalize_text(df)
df.head()

,review,sentiment
612,viva variety unique hybrid program parody trib...,positive
86,two year ago sundance loved josh kornbluth dir...,negative
662,would reccomend film everyone fan rocker lucia...,positive
965,movie true reflection australian resourcefulne...,positive
871,legend gala hollywood premiere screening space...,negative


In [7]:
df['sentiment'].value_counts()

sentiment
negative    260
positive    240
Name: count, dtype: int64

In [8]:
x = df['sentiment'].isin(['positive','negative'])
df = df[x]

In [9]:
df['sentiment'] = df['sentiment'].map({'positive':1, 'negative':0})
df.head()

,review,sentiment
612,viva variety unique hybrid program parody trib...,1
86,two year ago sundance loved josh kornbluth dir...,0
662,would reccomend film everyone fan rocker lucia...,1
965,movie true reflection australian resourcefulne...,1
871,legend gala hollywood premiere screening space...,0


In [10]:
df.isnull().sum()

review       0
sentiment    0
dtype: int64

In [11]:
vectorizer = CountVectorizer(max_features=100)
X = vectorizer.fit_transform(df['review'])
y = df['sentiment']

In [15]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.20, random_state=42)

In [13]:
import dagshub

mlflow.set_tracking_uri('https://dagshub.com/sarthak-ad011/Capstone.mlflow')
dagshub.init(repo_owner='sarthak-ad011', repo_name='Capstone', mlflow=True)

# mlflow.set_experiment("Logistic Regression Baseline")
mlflow.set_experiment("Logistic Regression Baseline")


❗❗❗ AUTHORIZATION REQUIRED ❗❗❗

/opt/anaconda3/envs/atlas/lib/python3.10/site-packages/rich/live.py:260: UserWarning: install "ipywidgets" for 
Jupyter support
  warnings.warn('install "ipywidgets" for Jupyter support')



Open the following link in your browser to authorize the client:
https://dagshub.com/login/oauth/authorize?state=474825f0-4093-432b-88b4-b81e6eecdacd&client_id=32b60ba385aa7cecf24046d8195a71c07dd345d9657977863b52e7748e0f0f28&middleman_request_id=3090ea5865ace39d52f0cca1e7962afe482a1a5bcdf06be92eeabfca08e80a5d




Accessing as sarthak-ad011

Initialized MLflow to track repo "sarthak-ad011/Capstone"

Repository sarthak-ad011/Capstone initialized!

2026/07/12 06:18:23 INFO mlflow.tracking.fluent: Experiment with name 'Logistic Regression Baseline' does not exist. Creating a new experiment.


<Experiment: artifact_location='mlflow-artifacts:/fa074756d7114693b382edc35800a872', creation_time=1783817303633, effective_trace_archival_retention=None, experiment_id='0', last_update_time=1783817303633, lifecycle_stage='active', name='Logistic Regression Baseline', tags={}, trace_location=None, workspace='default'>

In [16]:
import mlflow
import logging
import os
import time
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

# Configure logging
logging.basicConfig(level=logging.INFO, format="%(asctime)s - %(levelname)s - %(message)s")

logging.info("Starting MLflow run...")

with mlflow.start_run():
    start_time = time.time()
    
    try:
        logging.info("Logging preprocessing parameters...")
        mlflow.log_param("vectorizer", "Bag of Words")
        mlflow.log_param("num_features", 50)
        mlflow.log_param("test_size", 0.20)

        logging.info("Initializing Logistic Regression model...")
        model = LogisticRegression(max_iter=1000)  # Increase max_iter to prevent non-convergence issues

        logging.info("Fitting the model...")
        model.fit(X_train, y_train)
        logging.info("Model training complete.")

        logging.info("Logging model parameters...")
        mlflow.log_param("model", "Logistic Regression")

        logging.info("Making predictions...")
        y_pred = model.predict(X_test)

        logging.info("Calculating evaluation metrics...")
        accuracy = accuracy_score(y_test, y_pred)
        precision = precision_score(y_test, y_pred)
        recall = recall_score(y_test, y_pred)
        f1 = f1_score(y_test, y_pred)

        logging.info("Logging evaluation metrics...")
        mlflow.log_metric("accuracy", accuracy)
        mlflow.log_metric("precision", precision)
        mlflow.log_metric("recall", recall)
        mlflow.log_metric("f1_score", f1)

        logging.info("Saving and logging the model...")
        mlflow.sklearn.log_model(model, "model")

        # Log execution time
        end_time = time.time()
        logging.info(f"Model training and logging completed in {end_time - start_time:.2f} seconds.")

        # Save and log the notebook
        # notebook_path = "exp1_baseline_model.ipynb"
        # logging.info("Executing Jupyter Notebook. This may take a while...")
        # os.system(f"jupyter nbconvert --to notebook --execute --inplace {notebook_path}")
        # mlflow.log_artifact(notebook_path)

        # logging.info("Notebook execution and logging complete.")

        # Print the results for verification
        logging.info(f"Accuracy: {accuracy}")
        logging.info(f"Precision: {precision}")
        logging.info(f"Recall: {recall}")
        logging.info(f"F1 Score: {f1}")

    except Exception as e:
        logging.error(f"An error occurred: {e}", exc_info=True)


2026-07-12 06:24:19,528 - INFO - Starting MLflow run...
2026-07-12 06:24:20,084 - INFO - Logging preprocessing parameters...
2026-07-12 06:24:21,588 - INFO - Initializing Logistic Regression model...
2026-07-12 06:24:21,590 - INFO - Fitting the model...
2026-07-12 06:24:21,629 - INFO - Model training complete.
2026-07-12 06:24:21,629 - INFO - Logging model parameters...
2026-07-12 06:24:21,985 - INFO - Making predictions...
2026-07-12 06:24:21,987 - INFO - Calculating evaluation metrics...
2026-07-12 06:24:21,996 - INFO - Logging evaluation metrics...
2026-07-12 06:24:23,622 - INFO - Saving and logging the model...
2026/07/12 06:24:23 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026-07-12 06:24:36,766 - INFO - Model training and logging completed in 16.68 seconds.
2026-07-12 06:24:36,769 - INFO - Accuracy: 0.66
2026-07-12 06:24:36,770 - INFO - Precision: 0.6470588235294118
2026-07-12 06:24:36,771 - INFO - Recall: 0.673469387755102
2026-07-12 0

🏃 View run luminous-midge-918 at: https://dagshub.com/sarthak-ad011/Capstone.mlflow/#/experiments/0/runs/1283aab409884b6db5279bbd10d01782
🧪 View experiment at: https://dagshub.com/sarthak-ad011/Capstone.mlflow/#/experiments/0
